# Cell-centred HEST patch with Xenium transcripts

This notebook creates three clean, independently saved images for one Xenium cell:

1. the full **224 × 224 px at 0.5 µm/px (112 µm)** HEST field of view;
2. a tighter crop containing the centre cell and only its immediate neighbours; and
3. the same tight crop with the centre cell's high-confidence Xenium transcripts.

The image areas contain no titles, labels, legends, or scale-bar text. Xenium cell boundaries are yellow, the selected centre cell is red, and all transcripts use one shared cyan colour. The default (`TENX96`, cell `117333`) is a curated example. Change `SAMPLE_ID` and `CELL_ID` below to inspect another cell, and run the notebook in the `hest` environment.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
import zarr
from PIL import Image

# Curated default. CELL_ID is stored as a string so old integer and newer
# Xenium identifiers are handled consistently.
SAMPLE_ID = "TENX96"
CELL_ID = "117333"

PATCH_SIZE_PX = 224
PATCH_PIXEL_SIZE_UM = 0.5
ZOOM_SIZE_UM = 28
QV_THRESHOLD = 20

SEGMENTATION_COLOR = "#ECF010"  # yellow
CENTER_CELL_COLOR = "#EF3F17"  # red
# TRANSCRIPT_COLOR = "#00C8FF"  # one colour for every transcript
TRANSCRIPT_COLOR = "#EF3F17"  # one colour for every transcript
SEGMENTATION_LINE_WIDTH = 1.5
CENTER_CELL_LINE_WIDTH = 3.0
TRANSCRIPT_SIZE = 20

FIGURE_SIZE_IN = 5
SAVE_DPI = 240
SAVE_FIGURES = True


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "configs").is_dir() and (
            candidate / "src/cellbridge"
        ).is_dir():
            return candidate
    raise FileNotFoundError("Run from inside the cellbridge repository")


PROJECT_ROOT = find_project_root()
WORKSPACE_ROOT = PROJECT_ROOT.parents[1]
MANIFEST_PATH = WORKSPACE_ROOT / "data/train/hest/manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs/figures"

print(f"project:  {PROJECT_ROOT}")
print(f"manifest: {MANIFEST_PATH}")

In [ ]:
def polygon_parts(geometry):
    """Yield polygon parts from Polygon or MultiPolygon geometry."""
    if geometry.geom_type == "Polygon":
        yield geometry
    elif geometry.geom_type == "MultiPolygon":
        yield from geometry.geoms


def read_view(wsi, centre_xy, native_mpp, size_um, display_size_px=PATCH_SIZE_PX):
    """Read a square native-WSI crop and resize it for display."""
    source_size_px = round(size_um / native_mpp)
    x0 = round(centre_xy[0] - source_size_px / 2)
    y0 = round(centre_xy[1] - source_size_px / 2)
    x1, y1 = x0 + source_size_px, y0 + source_size_px

    height, width = wsi.shape[:2]
    if x0 < 0 or y0 < 0 or x1 > width or y1 > height:
        raise ValueError("Selected cell is too close to the WSI edge for this view")

    native_image = np.asarray(wsi[y0:y1, x0:x1, :3])
    image = np.asarray(
        Image.fromarray(native_image).resize(
            (display_size_px, display_size_px), Image.Resampling.LANCZOS
        )
    )
    return {
        "image": image,
        "x0": x0,
        "y0": y0,
        "x1": x1,
        "y1": y1,
        "scale": display_size_px / source_size_px,
        "display_size_px": display_size_px,
    }


def to_view_xy(xy, view):
    return (np.asarray(xy) - np.array([view["x0"], view["y0"]])) * view["scale"]


def draw_boundary(ax, geometry, view, *, color, linewidth, zorder):
    for polygon in polygon_parts(geometry):
        xy = to_view_xy(np.asarray(polygon.exterior.coords), view)
        ax.plot(
            xy[:, 0],
            xy[:, 1],
            color=color,
            linewidth=linewidth,
            solid_capstyle="round",
            solid_joinstyle="round",
            zorder=zorder,
        )


def draw_segmentations(ax, cells_in_view, target_geometry, view):
    for cell_id, geometry in cells_in_view.geometry.items():
        if cell_id != CELL_ID:
            draw_boundary(
                ax,
                geometry,
                view,
                color=SEGMENTATION_COLOR,
                linewidth=SEGMENTATION_LINE_WIDTH,
                zorder=2,
            )
    draw_boundary(
        ax,
        target_geometry,
        view,
        color=CENTER_CELL_COLOR,
        linewidth=CENTER_CELL_LINE_WIDTH,
        zorder=3,
    )


def clean_figure(image):
    """Return a borderless square figure whose axes contain only the image."""
    fig = plt.figure(figsize=(FIGURE_SIZE_IN, FIGURE_SIZE_IN), frameon=False)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(image, interpolation="nearest")
    size = image.shape[0]
    ax.set_xlim(0, size)
    ax.set_ylim(size, 0)
    ax.set_aspect("equal")
    ax.axis("off")
    return fig, ax


def save_figure(fig, filename):
    path = OUTPUT_DIR / filename
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=SAVE_DPI, pad_inches=0)
        print(f"saved: {path}")
    return path

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH, dtype={"id": str}).set_index("id")
if SAMPLE_ID not in manifest.index:
    raise KeyError(f"Unknown HEST sample: {SAMPLE_ID}")
sample = manifest.loc[SAMPLE_ID]
native_mpp = float(sample["pixel_size_um"])

# Cell polygons and transcripts are already registered to level-0 H&E pixels.
cells = gpd.read_parquet(sample["xenium_cell_seg"])
cells.index = cells.index.astype(str)
if CELL_ID not in cells.index:
    raise KeyError(f"Cell {CELL_ID} is not present in {SAMPLE_ID}")
target_geometry = cells.loc[CELL_ID, "geometry"]
centre = target_geometry.centroid
centre_xy = (centre.x, centre.y)

wsi_store = tifffile.imread(sample["wsi"], aszarr=True, level=0)
wsi = zarr.open(wsi_store, mode="r")
try:
    full_view = read_view(
        wsi,
        centre_xy,
        native_mpp,
        PATCH_SIZE_PX * PATCH_PIXEL_SIZE_UM,
    )
    zoom_view = read_view(wsi, centre_xy, native_mpp, ZOOM_SIZE_UM)
finally:
    wsi_store.close()

full_cells = cells.cx[
    full_view["x0"] : full_view["x1"], full_view["y0"] : full_view["y1"]
]
zoom_cells = cells.cx[
    zoom_view["x0"] : zoom_view["x1"], zoom_view["y0"] : zoom_view["y1"]
]

# Predicate pushdown reads only transcript row groups intersecting the zoom crop.
transcripts = pd.read_parquet(
    sample["transcripts"],
    columns=["cell_id", "feature_name", "he_x", "he_y", "qv"],
    filters=[
        ("he_x", ">=", zoom_view["x0"]),
        ("he_x", "<", zoom_view["x1"]),
        ("he_y", ">=", zoom_view["y0"]),
        ("he_y", "<", zoom_view["y1"]),
        ("qv", ">=", QV_THRESHOLD),
    ],
)
transcripts["cell_id"] = transcripts["cell_id"].astype(str)
transcripts["feature_name"] = transcripts["feature_name"].map(
    lambda value: value.decode("utf-8") if isinstance(value, bytes) else str(value)
)
control_prefixes = (
    "NegControlProbe_",
    "NegControlCodeword_",
    "BLANK_",
    "UnassignedCodeword_",
)
is_control = transcripts["feature_name"].str.startswith(control_prefixes, na=False)
target_transcripts = transcripts.loc[
    (transcripts["cell_id"] == CELL_ID) & ~is_control
].copy()
target_transcripts[["plot_x", "plot_y"]] = to_view_xy(
    target_transcripts[["he_x", "he_y"]].to_numpy(), zoom_view
)

print(
    f"{SAMPLE_ID} cell {CELL_ID}: {len(target_transcripts):,} QV≥{QV_THRESHOLD} "
    f"transcripts; {len(zoom_cells) - 1} neighbouring segmentations in the zoom view"
)

In [ ]:
# 1. Full HEST field of view: yellow Xenium boundaries, red centre cell.
full_fig, full_ax = clean_figure(full_view["image"])
draw_segmentations(full_ax, full_cells, target_geometry, full_view)
full_path = save_figure(
    full_fig, f"{SAMPLE_ID}_cell_{CELL_ID}_hest_patch.png"
)
plt.show()

# 2. Tight cell-and-neighbours crop with the same segmentation colours.
zoom_fig, zoom_ax = clean_figure(zoom_view["image"])
draw_segmentations(zoom_ax, zoom_cells, target_geometry, zoom_view)
zoom_path = save_figure(
    zoom_fig, f"{SAMPLE_ID}_cell_{CELL_ID}_zoom.png"
)
plt.show()

# 3. The same tight crop with every target-cell transcript in one colour.
transcript_fig, transcript_ax = clean_figure(zoom_view["image"])
draw_segmentations(transcript_ax, zoom_cells, target_geometry, zoom_view)
transcript_ax.scatter(
    target_transcripts["plot_x"],
    target_transcripts["plot_y"],
    s=TRANSCRIPT_SIZE,
    color=TRANSCRIPT_COLOR,
    alpha=0.9,
    edgecolor="white",
    linewidth=0.25,
    zorder=4,
)
transcript_path = save_figure(
    transcript_fig, f"{SAMPLE_ID}_cell_{CELL_ID}_zoom_transcripts.png"
)
plt.show()

The three PNGs are saved independently in `outputs/figures/`. Their image areas are deliberately annotation-free: yellow denotes neighbouring Xenium segmentations, red denotes the selected centre cell, and cyan denotes all non-control centre-cell transcripts passing `qv >= 20`. Adjust `ZOOM_SIZE_UM` if you want slightly more or less local context.